## Defining Improvement Over the Lag-1 Baseline

The lag-1 baseline achieved an overall MAE of approximately **1.11 units** over the 365-day evaluation period.

### Primary Metric

**MAE** will remain the primary evaluation metric because it:

- Measures error directly in units sold.
- Weights each unit of error equally.
- Is easy to interpret as the average forecast miss.

A new approach should achieve a lower overall MAE than lag-1.

### Supporting Evaluation

Overall MAE alone is not enough. Improvement should also be evaluated across meaningful product groupings to ensure that gains are not driven by a narrow subset of the data.

**RMSE** will remain a secondary metric to identify whether a model produces particularly large errors.

---

## 7-Day Seasonal Naive Forecast

The next forecasting approach will use demand from the same day of the previous week to predict daily demand:

$\hat{y}_t = y_{t-7}$

This replaces the lag-1 baseline's use of the previous day's demand ($t - 1$) with demand from the same day of the previous week ($t - 7$).

This provides a simple test of whether weekly shopping patterns improve forecasts over the lag-1 baseline while keeping the forecast horizon at one day.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from data.loader import load_raw_m5_data
from data.pipeline import build_item_store_day
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

RAW_DATA_PATH = Path("../data/raw")

In [2]:
item_store_day = build_item_store_day(*load_raw_m5_data(RAW_DATA_PATH))

In [3]:
forecasting_comparison = item_store_day[
    ["item_id", "store_id", "d", "date", "units_sold"]
].copy()

forecasting_comparison["lag_1_forecast"] = (
    forecasting_comparison
    .groupby(["item_id", "store_id"])["units_sold"]
    .shift(1)
)

forecasting_comparison["lag_7_forecast"] = (
    forecasting_comparison
    .groupby(["item_id", "store_id"])["units_sold"]
    .shift(7)
)

Creating `evaluation` containing the final 365 days from `forecasting_comparison`

In [4]:
evaluation = forecasting_comparison[
    forecasting_comparison["d"] > (forecasting_comparison["d"].max() - 365)
].copy()

In [5]:
# Check for exactly 365 days
evaluation["d"].nunique()

365

Calculate MAE for both predictions (lag-1 and lag-7)

In [6]:
item_store_summary = (
    evaluation
    .groupby(["item_id", "store_id"])
    .apply(
        lambda group: pd.Series(
            {
                "lag_1_mae": mean_absolute_error(
                    group["units_sold"],
                    group["lag_1_forecast"],
                ),
                "lag_7_mae": mean_absolute_error(
                    group["units_sold"],
                    group["lag_7_forecast"],
                )
            }
        ),
        include_groups=False
    )
    .reset_index()
)

In [7]:
# Should have expected cols and also 30,490 rows
item_store_summary.head()

,item_id,store_id,lag_1_mae,lag_7_mae
0,FOODS_1_001,CA_1,1.013699,0.920548
1,FOODS_1_001,CA_2,1.364384,1.391781
2,FOODS_1_001,CA_3,1.558904,1.591781
3,FOODS_1_001,CA_4,0.523288,0.512329
4,FOODS_1_001,TX_1,0.876712,0.871233


In [8]:
item_store_summary.shape

(30490, 4)

In [9]:
# Overall lag-1 and lag-7 MAEs
print("Lag-1 MAE")
print(
    mean_absolute_error(
        evaluation["units_sold"],
        evaluation["lag_1_forecast"],
    ),
    end="\n\n"
)
print("Lag-7 MAE")
print(
    mean_absolute_error(
        evaluation["units_sold"],
        evaluation["lag_7_forecast"],
    )
)

Lag-1 MAE
1.110958544683413

Lag-7 MAE
1.1528492162262947


In [10]:
# How many lag-7 predictions outperform their respective lag-1 predictions?
print(len(item_store_summary[
    item_store_summary["lag_7_mae"] < item_store_summary["lag_1_mae"]
]))
print(f"Out of total row count: {len(item_store_summary)}")

12512
Out of total row count: 30490


Lag-7 performs worse overall, but outperforms lag-1 for a substantial minority—about 41%—of individual item-store series.

In [11]:
print(
    "Lag-1 wins: "
    + str(
        len(
            item_store_summary[
                item_store_summary["lag_7_mae"] > item_store_summary["lag_1_mae"]
            ]
        )
    )
)
print(
    "Ties: "
    + str(
        len(
            item_store_summary[
                item_store_summary["lag_7_mae"] == item_store_summary["lag_1_mae"]
            ]
        )
    )
)

Lag-1 wins: 16846
Ties: 1132


lag-1 remains the better general-purpose baseline, but lag-7 wins often enough that weekly seasonality is worth investigating rather than dismissing.

### Findings
The naïve lag-7 approach outperforms the lag-1 baseline for 41% of item-store combinations, despite performing worse overall. \
This suggests that different forecasting approaches may be appropriate for different item-store combinations.

Next: investigate whether lag-7 improvements are randomly distributed or associated with product categories or store locations.